# Semana 6: La Derivada - Midiendo el Cambio
## Notebook de Ejercicios (NB2) - Dataset Diabetes

### Propósito de la sesión
Aplicar el concepto de derivada al entrenamiento de un modelo de regresión lineal usando **gradiente descendente**. Trabajaremos con el dataset **Diabetes** de scikit-learn, implementaremos el gradiente del MSE manualmente y visualizaremos cómo la recta de regresión se ajusta paso a paso.

### Objetivos de aprendizaje
*   Cargar y explorar el dataset **Diabetes** para regresión.
*   Derivar manualmente el gradiente del **MSE** para una regresión lineal simple.
*   Implementar el algoritmo de **gradiente descendente** desde cero.
*   Visualizar la evolución de la recta de regresión durante el entrenamiento.
*   Comparar los resultados con la solución analítica (ecuación normal) y con `sklearn`.

## Configuración Inicial

Importamos las librerías necesarias y cargamos el dataset Diabetes.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Configuración de estilo
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Cargamos el dataset Diabetes
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target
feature_names = diabetes.feature_names

print("🎯 Dataset Diabetes cargado correctamente.")
print(f"Shape de X: {X.shape}")
print(f"Shape de y: {y.shape}")
print(f"Características: {feature_names}")
print(f"Descripción del target: progresión de la enfermedad un año después")

### Exploración inicial de los datos

In [ ]:
# Creamos un DataFrame para visualizar
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print("Primeras 5 filas del dataset:")
display(df.head())

print("\nEstadísticas descriptivas:")
display(df.describe())

### Seleccionamos una característica para regresión simple

Para facilitar la visualización, trabajaremos con una sola característica. Según la literatura, la característica **bmi** (índice de masa corporal) suele tener buena correlación con el target.

In [ ]:
# Seleccionamos 'bmi' (índice de masa corporal) como característica
X_bmi = X[:, 2].reshape(-1, 1)  # 'bmi' está en el índice 2

# Dividimos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_bmi, y, test_size=0.2, random_state=42)

print(f"Shape X_train: {X_train.shape}")
print(f"Shape X_test: {X_test.shape}")

# Visualizamos la relación
plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, alpha=0.6, edgecolors='k', linewidth=0.5)
plt.xlabel('BMI (estandarizado)')
plt.ylabel('Progresión de la diabetes')
plt.title('Relación entre BMI y progresión de la diabetes')
plt.grid(True)
plt.show()

---
## Actividad 1: Derivar manualmente el gradiente del MSE

### 1.1. Modelo de regresión lineal simple

Nuestro modelo es:
$$\hat{y} = wx + b$$

### 1.2. Función de pérdida: MSE

Para un conjunto de $n$ muestras:
$$J(w, b) = \frac{1}{n} \sum_{i=1}^n (\hat{y}_i - y_i)^2 = \frac{1}{n} \sum_{i=1}^n (w x_i + b - y_i)^2$$

### 1.3. Derivadas parciales (gradiente)

Aplicando la regla de la cadena:

$$\frac{\partial J}{\partial w} = \frac{2}{n} \sum_{i=1}^n (w x_i + b - y_i) \cdot x_i$$

$$\frac{\partial J}{\partial b} = \frac{2}{n} \sum_{i=1}^n (w x_i + b - y_i)$$

En la práctica, a veces se omite el factor $2/n$ y se usa el error directamente, especialmente cuando se combina con la tasa de aprendizaje.

In [ ]:
# Implementamos las funciones de pérdida y gradiente
def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

def compute_gradients(X, y, w, b):
    """
    Calcula los gradientes de MSE respecto a w y b.
    X: array de características (n,)
    y: array de targets (n,)
    w, b: parámetros actuales
    """
    n = len(X)
    y_pred = w * X + b
    error = y_pred - y
    
    grad_w = (2/n) * np.sum(error * X)
    grad_b = (2/n) * np.sum(error)
    
    return grad_w, grad_b

# Probamos con valores iniciales
w_init = 0.0
b_init = 0.0
grad_w, grad_b = compute_gradients(X_train.flatten(), y_train, w_init, b_init)

print(f"Pesos iniciales: w={w_init}, b={b_init}")
print(f"Gradiente en w: {grad_w:.4f}")
print(f"Gradiente en b: {grad_b:.4f}")
print("📌 El gradiente nos indica la dirección y magnitud del cambio necesario.")

---
## Actividad 2: Implementar gradiente descendente

### 2.1. Algoritmo de gradiente descendente

Repetir hasta convergencia:
$$w := w - \eta \frac{\partial J}{\partial w}$$
$$b := b - \eta \frac{\partial J}{\partial b}$$

donde $\eta$ es la **tasa de aprendizaje** (learning rate).

In [ ]:
def gradient_descent(X, y, w_init=0.0, b_init=0.0, lr=0.1, epochs=100, verbose=True):
    """
    Gradiente descendente para regresión lineal simple.
    """
    w = w_init
    b = b_init
    n = len(X)
    
    # Historia para visualizar
    w_history = [w]
    b_history = [b]
    loss_history = [mse_loss(y, w*X + b)]
    
    for epoch in range(epochs):
        # Calcular gradientes
        grad_w, grad_b = compute_gradients(X, y, w, b)
        
        # Actualizar parámetros
        w = w - lr * grad_w
        b = b - lr * grad_b
        
        # Guardar historia
        w_history.append(w)
        b_history.append(b)
        loss_history.append(mse_loss(y, w*X + b))
        
        if verbose and (epoch % 20 == 0 or epoch == epochs-1):
            print(f"Epoch {epoch:3d}: w={w:.4f}, b={b:.4f}, loss={loss_history[-1]:.4f}")
    
    return w, b, w_history, b_history, loss_history

# Ejecutamos gradiente descendente
X_flat = X_train.flatten()
w_final, b_final, w_hist, b_hist, loss_hist = gradient_descent(
    X_flat, y_train, w_init=0.0, b_init=0.0, lr=0.5, epochs=100
)

print(f"\n✅ Parámetros finales: w={w_final:.4f}, b={b_final:.4f}")

### 2.2. Visualización de la convergencia

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(loss_hist, 'b-', linewidth=2)
plt.xlabel('Época')
plt.ylabel('MSE')
plt.title('Evolución de la pérdida durante el entrenamiento')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(w_hist, label='w', linewidth=2)
plt.plot(b_hist, label='b', linewidth=2)
plt.xlabel('Época')
plt.ylabel('Valor del parámetro')
plt.title('Evolución de los parámetros')
plt.legend()
plt.grid(True)

plt.suptitle('Convergencia del Gradiente Descendente', fontsize=14)
plt.tight_layout()
plt.show()

### 2.3. Visualización de la recta de regresión durante el entrenamiento

Veamos cómo cambia la recta a medida que avanzan las épocas.

In [ ]:
# Creamos puntos para dibujar la recta
x_line = np.linspace(X_train.min(), X_train.max(), 100).flatten()

plt.figure(figsize=(12, 8))

# Dibujamos los datos
plt.scatter(X_train, y_train, alpha=0.5, label='Datos de entrenamiento')

# Dibujamos la recta en diferentes épocas
epochs_to_plot = [0, 5, 20, 50, 99]
colors = ['gray', 'orange', 'green', 'blue', 'red']

for epoch, color in zip(epochs_to_plot, colors):
    w_epoch = w_hist[epoch]
    b_epoch = b_hist[epoch]
    y_line = w_epoch * x_line + b_epoch
    plt.plot(x_line, y_line, color=color, linewidth=2, 
             label=f'Época {epoch} (w={w_epoch:.2f}, b={b_epoch:.2f})')

plt.xlabel('BMI (estandarizado)')
plt.ylabel('Progresión de la diabetes')
plt.title('Evolución de la recta de regresión durante el gradiente descendente')
plt.legend()
plt.grid(True)
plt.show()

---
## Actividad 3: Comparación con solución analítica y sklearn

### 3.1. Solución analítica (ecuación normal)

Para regresión lineal simple, podemos calcular los parámetros óptimos directamente:

$$w = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}$$
$$b = \bar{y} - w \bar{x}$$

In [ ]:
# Solución analítica
x_mean = np.mean(X_flat)
y_mean = np.mean(y_train)

numerador = np.sum((X_flat - x_mean) * (y_train - y_mean))
denominador = np.sum((X_flat - x_mean)**2)
w_analitico = numerador / denominador
b_analitico = y_mean - w_analitico * x_mean

print("🔷 Solución analítica (ecuación normal):")
print(f"w = {w_analitico:.4f}")
print(f"b = {b_analitico:.4f}")

In [ ]:
# 3.2. Solución con sklearn
model_sk = LinearRegression()
model_sk.fit(X_train, y_train)

w_sk = model_sk.coef_[0]
b_sk = model_sk.intercept_

print("🔶 Solución con sklearn:")
print(f"w = {w_sk:.4f}")
print(f"b = {b_sk:.4f}")

# Comparación
print("\n📊 Comparación de métodos:")
print(f"{'Método':<20} {'w':<15} {'b':<15} {'MSE Test':<15}")
print("-" * 65)

# Predicciones en test
y_pred_gd = w_final * X_test.flatten() + b_final
y_pred_analitico = w_analitico * X_test.flatten() + b_analitico
y_pred_sk = model_sk.predict(X_test)

mse_gd = mean_squared_error(y_test, y_pred_gd)
mse_analitico = mean_squared_error(y_test, y_pred_analitico)
mse_sk = mean_squared_error(y_test, y_pred_sk)

print(f"{'Gradiente Desc.':<20} {w_final:<15.4f} {b_final:<15.4f} {mse_gd:<15.4f}")
print(f"{'Ecuación Normal':<20} {w_analitico:<15.4f} {b_analitico:<15.4f} {mse_analitico:<15.4f}")
print(f"{'sklearn':<20} {w_sk:<15.4f} {b_sk:<15.4f} {mse_sk:<15.4f}")

### 3.3. Visualización de las tres rectas

In [ ]:
plt.figure(figsize=(12, 8))

# Datos
plt.scatter(X_train, y_train, alpha=0.4, label='Datos entrenamiento')
plt.scatter(X_test, y_test, alpha=0.6, marker='s', label='Datos prueba', edgecolors='k')

# Rectas
x_line = np.linspace(X_train.min(), X_train.max(), 100).flatten()

plt.plot(x_line, w_final * x_line + b_final, 'r-', linewidth=2, label=f'Gradiente Desc. (w={w_final:.2f})')
plt.plot(x_line, w_analitico * x_line + b_analitico, 'b--', linewidth=2, label=f'Ec. Normal (w={w_analitico:.2f})')
plt.plot(x_line, w_sk * x_line + b_sk, 'g:', linewidth=2, label=f'sklearn (w={w_sk:.2f})')

plt.xlabel('BMI (estandarizado)')
plt.ylabel('Progresión de la diabetes')
plt.title('Comparación de métodos: Gradiente Descendente vs Ecuación Normal vs sklearn')
plt.legend()
plt.grid(True)
plt.show()

---
## Actividad 4: Experimentación con hiperparámetros

### 4.1. Efecto de la tasa de aprendizaje (learning rate)

In [ ]:
learning_rates = [0.01, 0.1, 0.5, 1.0, 2.0]
epochs = 50

plt.figure(figsize=(14, 6))

for lr in learning_rates:
    _, _, _, _, loss_hist_lr = gradient_descent(
        X_flat, y_train, w_init=0.0, b_init=0.0, lr=lr, epochs=epochs, verbose=False
    )
    plt.plot(loss_hist_lr, label=f'lr={lr}', linewidth=2)

plt.xlabel('Época')
plt.ylabel('MSE')
plt.title('Efecto de la tasa de aprendizaje en la convergencia')
plt.legend()
plt.grid(True)
plt.ylim(0, 10000)
plt.show()

print("📌 Observa: lr muy pequeño converge lentamente, lr muy grande puede divergir.")

### 4.2. Efecto de la inicialización

In [ ]:
inits = [(0.0, 0.0), (2.0, 50.0), (-2.0, -50.0)]
epochs = 100
lr = 0.5

plt.figure(figsize=(14, 6))

for w0, b0 in inits:
    w_f, b_f, w_h, b_h, loss_h = gradient_descent(
        X_flat, y_train, w_init=w0, b_init=b0, lr=lr, epochs=epochs, verbose=False
    )
    plt.plot(loss_h, label=f'w0={w0}, b0={b0}', linewidth=2)

plt.xlabel('Época')
plt.ylabel('MSE')
plt.title('Efecto de la inicialización en la convergencia')
plt.legend()
plt.grid(True)
plt.ylim(0, 10000)
plt.show()

print("📌 La inicialización afecta la velocidad de convergencia, pero para funciones convexas como MSE, todos convergen al mismo mínimo.")

---
## Ejercicios para el Estudiante

1.  **Otra característica:** Repite todo el proceso pero usando la característica **bp** (presión sanguínea) en lugar de BMI. ¿Qué diferencia observas en la convergencia y en el error final?

2.  **Múltiples características:** Extiende la implementación de gradiente descendente para usar **todas las características** del dataset (regresión múltiple). Compara el tiempo de ejecución y el error final.

3.  **Mini-batch Gradient Descent:** Modifica el algoritmo para que use **mini-batches** (por ejemplo, batches de 32 muestras) en lugar de todo el dataset. Compara la velocidad de convergencia y el comportamiento del loss.

4.  **Regularización:** Añade un término de regularización L2 (Ridge) a la función de pérdida y deriva el nuevo gradiente. Implementa **Ridge Regression** con gradiente descendente.

5.  **Reflexión:** ¿Por qué el gradiente descendente es preferible a la ecuación normal cuando el número de características es muy grande (ej. > 10,000)? ¿Qué limitaciones tiene cada método?

---
## Conclusiones de la Sesión

*   Hemos **derivado manualmente** el gradiente del MSE para regresión lineal simple, comprendiendo su origen matemático.
*   Implementamos el algoritmo de **gradiente descendente** desde cero y visualizamos cómo la recta de regresión se ajusta paso a paso.
*   Comparamos nuestros resultados con la **solución analítica** (ecuación normal) y con la implementación de **sklearn**, verificando que todos convergen a la misma solución.
*   Experimentamos con diferentes **tasas de aprendizaje** e **inicializaciones**, observando su impacto en la convergencia.
*   Este ejercicio consolida la conexión entre la **derivada** (como herramienta de optimización) y el entrenamiento real de modelos de machine learning.

¡Ahora entiendes cómo la derivada guía a los modelos hacia el mínimo del error!